# Chatroom App

Here we create a multi-user **Chatroom App** using Flet. The user messages are broadcasted via Flet's built-in PubSub library. Moreover, we will show how to enhance the UI with custom styled controls. Finally, we show how to trivially (in-memory) share state between sessions, such as chat history & user info. This app is a good starting point for creating more serious projects.

## Broadcasting with PubSub

<video
  src="./img/flet-chat/v1.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

This run using `flet run --web v1.py` and we open the same link in separate browsers creating two sessions.

```python
import flet as ft
from dataclasses import dataclass

@dataclass
class Message:
    user: str
    text: str

@ft.component
def AppView():
    page = ft.context.page
    session_id = page.session.id
    history, set_history = ft.use_state([])     # <1>
    message, set_message = ft.use_state("")
    
    def on_message(msg_obj: Message):           # <3>
        page.run_thread(lambda: set_history(lambda h: [*h, msg_obj]))
        page.update()

    # subscribe once. use_effect expects cleanup function
    def subscribe():                                        
        page.pubsub.subscribe(on_message)
        def cleanup(): 
            page.pubsub.unsubscribe(on_message)
        return cleanup

    # empty deps => run once on mount, and cleanup on unmount
    ft.use_effect(subscribe, [])        # <2>

    def send_click(e):  # <4>
        page.pubsub.send_all(msg_obj=Message(user=session_id, text=message))
        set_message("")

    return ft.Column(
        controls=[
            ft.Column(controls=[ft.Text(f"{m.user}: {m.text}") for m in history]),
            ft.Row(controls=[
                ft.TextField(
                    label="New message",
                    value=message,
                    width=400,
                    on_change=lambda e: set_message(e.control.value),
                    on_submit=send_click
                ),
                ft.Button("Send", on_click=send_click)
            ]),
        ]
    )


if __name__ == "__main__":
    ft.run(lambda page: page.render(AppView))
```

1. The history and current message are initialized using `use_state` hook. So that these are persisted across re-renders. 

2. Then, the "effect" `subscribe` is called only once during initialization since the dependencies are empty `[]`. See the docs on [`use_effect`](https://docs.flet.dev/types/useeffect/?h=use_effect). This also returns a cleanup function which unsubscribes from pubsub. The subscribe mechanism makes a FastAPI worker process trigger the function `on_message`  whenever a message is published. Moreover, this works due to having closure on `page`.

3. This calls `set_history` with an update function (note that we can either put a value or an update function with the state variable as input). Here we opted for an update function to ensure that the latest message is captured. Finally, we force update. This is run in the current page's thread (sync)[^run_thread].

4. We've set up the plumbing for getting messages. How about sending? For this we simply use `page.pubsub.send_all`. This takes in any Python object and becomes the input of the functions that was specified during PubSub subscribe. This explains the input type `msg_obj: Message` for `on_message`. We likewise send an object of type `Message`. Finally, the `message` variable is cleared.

[^run_thread]: The exact mechanism is unclear but this is what finally worked to properly run everything in the current render context without UX issues, and no error logs.

## Adding usernames

Here we add usernames which users choose when they join the chatroom. This naturally has to be unique. Hence, we have an `active_users` set of strings. Note that since we have multi-users and the app works in a distributed manner, this adds a layer of complexity in ensuring consistency. In particular, we're wary of **race conditions.** For example, we want to validate that a username is unique, however by the time we have completed creating a user another may have registered the same username, resulting in duplicate usernames.

To solve this, we implement **thread locks** ensuring that only one thread accesses `active_users` at a time.

```python
@dataclass
class ChatRoom:
    active_users: set[str] = field(default_factory=set)
    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)

    def add_user(self, name: str):
        with self._lock:
            self.validate_username(name)
            self.active_users.add(name)

    def remove_user(self, name: str):
        with self._lock:
            self.active_users.discard(name)

    def validate_username(self, name: str):
        if name in self.active_users:
            raise ValueError(f'"{name}" is already taken. Please choose another.')
        if not name.strip():
            raise ValueError("Username cannot be empty.")
```

Chatroom [state is shared]{.mark} by users by bootstrapping a `ChatRoom` instance to all pages:

```python
if __name__ == "__main__":
    # shared state across sessions
    chatroom = ChatRoom()

    def bootstrap(page: ft.Page):
        page.chat = chatroom
        page.render(AppView)

    ft.run(bootstrap)
```

Next, we have the **join dialog**. To understand this, let's walk backwards:

```python
def JoinDialog(join_click: Callable, chatroom: ChatRoom):
    def join_click_loop():                                                          # <1>
        def handler(e):
            try:
                e.page.pop_dialog()
                entered = username.value.strip()
                chatroom.add_user(entered)
                join_click(e, entered)                                              # <2>

            except ValueError as error:                                             # <3>
                e.page.pop_dialog()
                e.page.show_dialog(
                    ft.AlertDialog(
                        modal=True,
                        title=ft.Text("Invalid Username"),
                        content=ft.Text(str(error)),
                        actions=[
                            ft.Button(
                                "OK", 
                                on_click=lambda _: (                    
                                    e.page.pop_dialog(),                                    # pop error dialog        
                                    e.page.show_dialog(JoinDialog(join_click, chatroom))    # start over with a fresh join dialog
                                )
                            )
                        ],
                        actions_alignment=ft.MainAxisAlignment.END,
                    )
                )
                return
        return handler
    
    username = ft.TextField(
        label="Enter your name",
        on_submit=join_click_loop(),
        autofocus=True,
    )

    return ft.AlertDialog(
        modal=True, 
        title=ft.Text("Welcome!"),
        content=ft.Column([username], tight=True),
        actions=[ft.Button("Join", on_click=join_click_loop())], 
        actions_alignment=ft.MainAxisAlignment.END
    )
```
1. This first defines a loop for the handler `join_click` to keep creating and deleting a dialog until we get a valid one. 
Eech branch of the loop starts with a `pop_dialog` for the returned modal. Both ENTER and pressing the "Join" button results
in entering the loop.
2. Happy path ultimately calls `join_click(e, entered)` where `entered` is a valid username.
3. In case username already exists, we get a `ValueError`. Then, the join dialog is closed, an error dialog is shown,
and a new loop is started. Note that the error dialog is non-blocking, so we have to exit the handler
by using `return`.

The main difference in the app is that the join dialog is opened at start up. Hence we upgrade `subscribe` to `join_and_subscribe`:

```python
    def join_and_subscribe():
        page.pubsub.subscribe(on_message)

        try:
            stored_username = page.session.store.get("username") or ""
            page.chat.add_user(stored_username)
            set_username(stored_username)
            
        except ValueError:
            def on_join(e, entered_name):
                set_username(entered_name)
                page.session.store.set("username", entered_name)
                page.pubsub.send_all(
                    Message(
                        user="System", 
                        text=f"{entered_name} joined the chat!"
                    )
                )

            page.show_dialog(JoinDialog(on_join, page.chat))

        def cleanup():
            current_user = page.session.store.get("username") or ""
            page.chat.remove_user(current_user)
            page.pubsub.unsubscribe(on_message)

        return cleanup

    ft.use_effect(join_and_subscribe, [])
```

<video
  src="./img/flet-chat/v2.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  onloadeddata="this.playbackRate=1.25"
  style="max-width:100%;">
</video>

## Enhancing the UI

### Chat messages

When a user joins, we print the message that a user has joined in italicized gray:

```{.python filename="src/v3.py"}
@ft.control
class ChatMessage(ft.Row):
    def __init__(self, message: Message):
        super().__init__()
        self.message = message
        self.vertical_alignment = ft.CrossAxisAlignment.START
        self.controls = [
            ft.CircleAvatar(
                content=ft.Text(self.get_initials(self.message.user)),
                color=ft.Colors.WHITE,
                bgcolor=self.get_avatar_color(self.message.user),
            ),
            ft.Column(
                tight=True,
                spacing=5,
                controls=[
                    ft.Text(self.message.user, weight=ft.FontWeight.BOLD),
                    ft.Text(self.message.text, selectable=True),
                ],
            ),
        ]

    def get_initials(self, username: str):
        if username:
            return username[:1].capitalize()
        else:
            return "?"

    def get_avatar_color(self, username: str):
        colors_lookup = [
            ft.Colors.AMBER,
            ft.Colors.BLUE,
            ft.Colors.BROWN,
            ft.Colors.CYAN,
            ft.Colors.GREEN,
            ft.Colors.INDIGO,
            ft.Colors.LIME,
            ft.Colors.ORANGE,
            ft.Colors.PINK,
            ft.Colors.PURPLE,
            ft.Colors.RED,
            ft.Colors.TEAL,
            ft.Colors.YELLOW,
        ]
        color_idx = hash(username) % len(colors_lookup)
        return colors_lookup[color_idx]


def build_messages(messages: list[Message]) -> list[ft.Control]:
    controls = []
    for msg in messages:
        if msg.user == "System":
            text = msg.text
            system_message = ft.Text(text, italic=True, color=ft.Colors.GREY)
            controls.append(system_message)
        else:
            controls.append(ChatMessage(msg))
    return controls
```

:::{.callout-caution}
Setting `color=ft.colors.GREY` or `color=ft.Colors.GRAY` by mistake silently fails, i.e. the messages are not shown in the UI but no exception is raised! IDE autocomplete and syntax highlighting are crucial to ensure that we are using an existing object.
:::

### Arranging the chat panel

The page is a column consisting of a **container** (chat history) and a **row** (new message). The container contains a `ListView` to get a scrollable list of contents. Then, each chat message is a `ChatMessage` control defined above, or a `TextField` containing system messages. The new message row consists of a `TextField` and a send icon. We just have to ensure that the controls are extended and aligned into appropriate positions and proportions.

![**Anatomy of our chat app UI.** It's good practice to draw a design like this by hand before coding.](./img/flet-chat/page-layout.svg)

Here we change `send_click` to async to allow for **autofocus**. We similarly add autofocus to new message `TextField` and the join `TextField` so that the UX is seamless. This small change is very important, otherwise the users will have to click the message field each time they want to send a message. We also take advantage of `Container` to wrap the list of messages. Using `ListView` with **autoscroll** means that when the number of messages is large enough to fill the container it will do autoscroll animation so the last message is viewable. This is another critical factor for the UX. Finally, `expand=True` is taken advantage of everywhere.

```{.python filename="src/v3.py"}
...
@ft.component
def AppView():
    ...

    async def send_click(e):
        if message.strip() and username:
            page.pubsub.send_all(Message(user=username, text=message))
            set_message("")
            await new_message.focus()

    chat = ft.ListView(
        controls=build_messages(history),
        expand=True,
        spacing=10,
        auto_scroll=True,
    )

    new_message = ft.TextField(
        label="New message",
        value=message,
        expand=True,
        on_change=lambda e: set_message(e.control.value),
        on_submit=send_click,
        autofocus=True,
    )

    new_message_row = ft.Row(
        controls=[
            new_message,
            ft.Button(content=ft.Icon(ft.Icons.SEND), on_click=send_click)
        ],
    )

    return ft.Column(
        controls=[
            ft.Container(
                content=chat,
                border=ft.Border.all(1, ft.Colors.OUTLINE),
                border_radius=5,
                padding=10,
                expand=True,
            ),
            new_message_row
        ],
        expand=True,
    )
```

<br>

**Final demo.** This demo involves three users. Autoscroll and autofocus are emphasized, along with earlier features:

<video
  src="./img/flet-chat/v3.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  onloadeddata="this.playbackRate=1.5"
  style="max-width:100%;">
</video>

<br>

**Source**

In [ ]:
#| code-fold: true
import flet as ft
import threading

from typing import Callable, Optional
from dataclasses import dataclass, field


@dataclass
class ChatRoom:
    active_users: set[str] = field(default_factory=set)
    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)

    def add_user(self, name: str):
        with self._lock:
            self.validate_username(name)
            self.active_users.add(name)

    def remove_user(self, name: str):
        with self._lock:
            self.active_users.discard(name)

    def validate_username(self, name: str):
        if name in self.active_users:
            raise ValueError(f'"{name}" is already taken. Please choose another.')
        if not name.strip():
            raise ValueError("Username cannot be empty.")


def JoinDialog(join_click: Callable, chatroom: ChatRoom):
    def join_click_loop():                                                          # <1>
        def handler(e):
            try:
                e.page.pop_dialog()
                entered = username.value.strip()
                chatroom.add_user(entered)
                join_click(e, entered)                                              # <2>

            except ValueError as error:                                             # <3>
                e.page.pop_dialog()
                e.page.show_dialog(
                    ft.AlertDialog(
                        modal=True,
                        title=ft.Text("Invalid Username"),
                        content=ft.Text(str(error)),
                        actions=[
                            ft.Button(
                                "OK", 
                                on_click=lambda _: (                    
                                    e.page.pop_dialog(),                                    # pop error dialog        
                                    e.page.show_dialog(JoinDialog(join_click, chatroom))    # start over with a fresh join dialog
                                )
                            )
                        ],
                        actions_alignment=ft.MainAxisAlignment.END,
                    )
                )
                return
        return handler
    
    username = ft.TextField(
        label="Enter your name",
        on_submit=join_click_loop(),
        autofocus=True,
    )

    return ft.AlertDialog(
        modal=True, 
        title=ft.Text("Welcome!"),
        content=ft.Column([username], tight=True),
        actions=[ft.Button("Join", on_click=join_click_loop())], 
        actions_alignment=ft.MainAxisAlignment.END
    )


@dataclass
class Message:
    user: str
    text: str


@ft.control
class ChatMessage(ft.Row):
    def __init__(self, message: Message):
        super().__init__()
        self.message = message
        self.vertical_alignment = ft.CrossAxisAlignment.START
        self.controls = [
            ft.CircleAvatar(
                content=ft.Text(self.get_initials(self.message.user)),
                color=ft.Colors.WHITE,
                bgcolor=self.get_avatar_color(self.message.user),
            ),
            ft.Column(
                tight=True,
                spacing=5,
                controls=[
                    ft.Text(self.message.user, weight=ft.FontWeight.BOLD),
                    ft.Text(self.message.text, selectable=True),
                ],
            ),
        ]

    def get_initials(self, username: str):
        if username:
            return username[:1].capitalize()
        else:
            return "?"

    def get_avatar_color(self, username: str):
        colors_lookup = [
            ft.Colors.AMBER,
            ft.Colors.BLUE,
            ft.Colors.BROWN,
            ft.Colors.CYAN,
            ft.Colors.GREEN,
            ft.Colors.INDIGO,
            ft.Colors.LIME,
            ft.Colors.ORANGE,
            ft.Colors.PINK,
            ft.Colors.PURPLE,
            ft.Colors.RED,
            ft.Colors.TEAL,
            ft.Colors.YELLOW,
        ]
        color_idx = hash(username) % len(colors_lookup)
        return colors_lookup[color_idx]


def build_messages(messages: list[Message]) -> list[ft.Control]:
    controls = []
    for msg in messages:
        if msg.user == "System":
            text = msg.text
            system_message = ft.Text(text, italic=True, color=ft.Colors.GREY)
            controls.append(system_message)
        else:
            controls.append(ChatMessage(msg))
    return controls


@ft.component
def AppView():
    page = ft.context.page
    username, set_username = ft.use_state("")
    history, set_history = ft.use_state([])
    message, set_message = ft.use_state("")

    def on_message(msg_obj: Message):
        page.run_thread(lambda: set_history(lambda h: [*h, msg_obj]))
        page.update()

    def join_and_subscribe():
        page.pubsub.subscribe(on_message)

        try:
            stored_username = page.session.store.get("username") or ""
            page.chat.add_user(stored_username)
            set_username(stored_username)
            
        except ValueError:
            def on_join(e, entered_name):
                set_username(entered_name)
                page.session.store.set("username", entered_name)
                page.pubsub.send_all(
                    Message(
                        user="System", 
                        text=f"{entered_name} joined the chat!"
                    )
                )

            page.show_dialog(JoinDialog(on_join, page.chat))

        def cleanup():
            current_user = page.session.store.get("username") or ""
            page.chat.remove_user(current_user)
            page.pubsub.unsubscribe(on_message)

        return cleanup

    ft.use_effect(join_and_subscribe, [])

    async def send_click(e):
        if message.strip() and username:
            page.pubsub.send_all(Message(user=username, text=message))
            set_message("")
            await new_message.focus()

    chat = ft.ListView(
        controls=build_messages(history),
        expand=True,
        spacing=10,
        auto_scroll=True,
    )

    new_message = ft.TextField(
        label="New message",
        value=message,
        expand=True,
        on_change=lambda e: set_message(e.control.value),
        on_submit=send_click,
        autofocus=True,
    )

    new_message_row = ft.Row(
        controls=[
            new_message,
            ft.Button(content=ft.Icon(ft.Icons.SEND), on_click=send_click)
        ],
    )

    return ft.Column(
        controls=[
            ft.Container(
                content=chat,
                border=ft.Border.all(1, ft.Colors.OUTLINE),
                border_radius=5,
                padding=10,
                expand=True,
            ),
            new_message_row
        ],
        expand=True,
    )


if __name__ == "__main__":
    # shared state across sessions
    chatroom = ChatRoom()

    def bootstrap(page: ft.Page):
        page.chat = chatroom
        page.render(AppView)

    ft.run(bootstrap)